# File configs.yaml

In [1]:
# %%writefile configs.yaml
# model:
#     hidden_dim: 1024
#     num_dit_blocks: 22
#     head_dim: 64
#     num_heads: 16
#     text_dim: 512
#     ff_mult: 4
#     dropout: 0.0
#     mel_dim: 100
#     vocab_size: 71
#     text_embedding_type: "convnext"
#     text_conformer_layers: 1
#     text_conformer_heads: 4
#     text_convnext_layers: 4
#     pe_attn_head:
#     text_mask_padding: True
#     vocab_path: "/kaggle/input/datasets/huutrank4ds/viflow-vocab/vocab.txt"

# data:
#     dataset_dirs:
#         - "/kaggle/input/datasets/huuvahan/melspecviflow"
#     val_sample_path: "/kaggle/input/datasets/huutrank4ds/validation-sample-viflow/mixed_speaker_samples.csv"

# mel:
#     sample_rate: 24000
#     n_fft: 1024
#     hop_length: 256
#     win_length: 1024
#     n_mels: 100
#     fmin: 0
#     fmax: 12000.0
#     power: 1.0
#     mel_scale: "slaney"
#     norm: "slaney"
#     log_clip_min: 1.0e-5

# train:
#     num_epoch_kaggle_version: 20
#     port: '12355'
#     val_batch_size: 32
#     max_frames: 70000
#     num_workers: 2
#     learning_rate: 1e-4
#     weight_decay: 1.0e-2
#     warmup_steps: 5000
#     total_steps: 150000
#     grad_clip_norm: 1.0
#     num_epochs: 150
#     grad_accum_steps: 1
#     grad_clip: 1
#     checkpoint_dir: "checkpoints"
#     log_dir: "logs"
#     resume_path: "/kaggle/input/models/huutrank4ds/viflow-full-gear/pytorch/default/9/viflow_epoch_104.pt"
#     use_amp: True
#     use_scheduler: True
#     p_drop_audio_cond: 0.15
#     p_drop_text: 0.15

# inference:
#     ode_steps: 32

# Đặt seed cho toàn notebook

In [2]:
import torch
import numpy as np
import random
import os

def seed_everything(seed=35):
    # 1. Định nghĩa biến môi trường cho Python Hash (quan trọng cho các cấu trúc như set, dict)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 2. Seed cho Python core
    random.seed(seed)
    
    # 3. Seed cho NumPy
    np.random.seed(seed)
    
    # 4. Seed cho PyTorch (CPU và tất cả GPU)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # Nếu dùng multi-GPU
    
    # 5. Cấu hình cho cuDNN (Xương sống của tính toán GPU)
    # Buộc cuDNN dùng các thuật toán mang tính định nghĩa (deterministic)
    torch.backends.cudnn.deterministic = True
    
    # Tắt tính năng tự động tìm thuật toán nhanh nhất (benchmark) 
    # Vì thuật toán nhanh nhất có thể thay đổi và gây ra sai lệch nhỏ về số thập phân
    torch.backends.cudnn.benchmark = False
    
    # 6. Ép PyTorch sử dụng các thuật toán deterministic nếu có thể
    # Lưu ý: Một số hàm có thể báo lỗi nếu chưa có bản deterministic, lúc đó hãy cân nhắc
    # torch.use_deterministic_algorithms(True)
    
    print(f"✅ Đã thiết lập môi trường định tính với Seed: {seed}")

# Chạy lệnh này ở đầu Notebook
seed_everything(35)

✅ Đã thiết lập môi trường định tính với Seed: 35


# Tạo file vocab

In [3]:
import os
from pathlib import Path
import h5py
from tqdm import tqdm

def build_vocab_from_h5_dirs(directories, save_path="vocab.txt"):
    """
    Quét danh sách các thư mục để tìm file .h5, 
    đọc attribute 'phonemes' từ tất cả các mẫu bên trong để tạo từ vựng.
    """
    h5_files = []
    
    print("Đang quét các thư mục để tìm file .h5...")
    for directory in directories:
        # Dùng rglob để tìm đệ quy (bao gồm cả thư mục con)
        # Nếu file h5 của bạn không nằm trong thư mục con, có thể đổi thành glob("*.h5")
        found_files = list(Path(directory).rglob("*.h5"))
        h5_files.extend(found_files)
        print(f" - {directory}: {len(found_files)} files")

    if not h5_files:
        print("Không tìm thấy file .h5 nào trong các thư mục đã cho.")
        return

    print(f"Tổng số file .h5 cần xử lý: {len(h5_files)}")
    
    symbols = set()

    for h5_path in tqdm(h5_files, desc="Đang trích xuất âm vị"):
        try:
            with h5py.File(
                h5_path,
                "r",
                libver="latest",
                swmr=True,
                rdcc_nbytes=0,
                rdcc_nslots=1,
            ) as f:
                
                # f.keys() trả về danh sách các sample_id bên trong file h5
                for sample_id in f.keys():
                    phonemes = f[sample_id].attrs.get("phonemes", "")
                    
                    # h5py đôi khi trả về kiểu bytes thay vì string, cần decode an toàn
                    if isinstance(phonemes, bytes):
                        phonemes = phonemes.decode("utf-8", errors="ignore")
                    else:
                        phonemes = str(phonemes)
                        
                    symbols.update(list(phonemes))
                    
        except Exception as e:
            # Bỏ qua file lỗi nhưng vẫn in ra cảnh báo để bạn kiểm tra sau
            print(f"\n[Cảnh báo] Lỗi khi đọc file {h5_path}: {e}")

    # Sắp xếp vocab theo thứ tự từ điển
    vocab_list = sorted(symbols)

    with open(save_path, "w", encoding="utf-8") as f:
        for s in vocab_list:
            f.write(f"{s}\n")

    print(f"\nĐã hoàn thành! Lưu thành công {len(vocab_list)} ký hiệu duy nhất vào: {save_path}")

# Tải mã nguồn

In [4]:
!git clone https://github.com/huutrank4ds/ViFlow.git -q

import sys
sys.path.append("/kaggle/working/ViFlow") 

# Trỏ chính xác vào file requirements.txt bên trong thư mục vừa clone
!pip install -r ViFlow/requirements.txt -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 78.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.5 MB/s eta 0:00:00


# Đọc file config

In [5]:
import os
import yaml

def load_config(config_path):
    """Đọc file cấu hình và trả về dictionary"""
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"❌ Không tìm thấy file cấu hình tại: {config_path}")
        
    with open(config_path, 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)
    return config

# Load dataset

In [6]:
from dataset import VietnamesePhonemeTokenizer, ViFlowH5Dataset, ViFlowCollate, prepare_viflow_cache, load_viflow_metadata

cfg = load_config("/kaggle/working/ViFlow/configs.yaml")
cfg["data"]["dataset_dirs"] = ["/kaggle/input/datasets/huuvahan/melspecviflow"] # Các thư mục chứa các file hdf5
cfg["data"]["val_sample_path"] = "/kaggle/input/datasets/huutrank4ds/validation-sample-viflow/mixed_speaker_samples.csv" # File chứa id các mẫu thuộc tập validation

In [7]:
build_vocab_from_h5_dirs(cfg["data"]["dataset_dirs"])

Đang quét các thư mục để tìm file .h5...
 - /kaggle/input/datasets/huuvahan/melspecviflow: 10 files
Tổng số file .h5 cần xử lý: 10


Đang trích xuất âm vị: 100%|██████████| 10/10 [05:37<00:00, 33.74s/it]


Đã hoàn thành! Lưu thành công 69 ký hiệu duy nhất vào: vocab.txt


# Training

In [ ]:
# ======================================================================================
# DANH SÁCH CÁC THAM SỐ LỆNH (COMMAND LINE ARGUMENTS) ĐỂ GHI ĐÈ FILE CONFIGS.YAML
# ======================================================================================
# --config:            Đường dẫn đến file cấu hình gốc (mặc định: configs.yaml)
# --use_scheduler:     Có sử dụng Learning Rate Scheduler hay không (true/false)
# --max_frames:        Số lượng frame tối đa cho thuật toán chia batch động (Dynamic Batching)
# --warmup_steps:      Số bước tăng dần tốc độ học (warmup steps) cho Scheduler
# --total_steps:       Tổng số bước (steps) cho toàn bộ quá trình huấn luyện
# --resume_path:       Đường dẫn tới file checkpoint cũ để tiếp tục huấn luyện
# --vocab_path:        Đường dẫn tới file vocab.txt chứa bộ từ vựng (âm vị)
# --learning_rate:     Tốc độ học (learning rate) tối đa
# --grad_accum_steps:  Số bước tích lũy gradient trước khi thực sự cập nhật trọng số mô hình
# --grad_clip:         Giá trị ngưỡng cắt xén (clip) gradient để tránh lỗi bùng nổ gradient
# --num_epochs:        Tổng số lượng kỷ nguyên (epoch) huấn luyện
# --p_drop_audio_cond: Xác suất loại bỏ điều kiện âm thanh (áp dụng cho Classifier-Free Guidance)
# --p_drop_text:       Xác suất loại bỏ điều kiện văn bản (áp dụng cho Classifier-Free Guidance)
# --use_amp:           Có sử dụng huấn luyện độ chính xác hỗn hợp (AMP) để giảm VRAM hay không (true/false)
# ======================================================================================

%cd ViFlow
!python train.py --max_frames 2000 --grad_accum_steps 20

[Errno 2] No such file or directory: 'ViFlow'
/kaggle/working/ViFlow
=== CONFIGURATION OVERRIDES ===
Scheduler    : True
Max Frames   : 2000
Warmup Steps : 5000
Total Steps  : 150000
Learning Rate: 1e-4
Resume Path  : /kaggle/input/models/huutrank4ds/viflow-full-gear/pytorch/default/9/viflow_epoch_104.pt
Vocab Path   : /kaggle/input/datasets/huutrank4ds/viflow-vocab/vocab.txt
Grad Accum   : 20
Grad Clip    : 1
Num Epochs   : 150
P Drop Audio  : 0.15
P Drop Text   : 0.15
Use AMP      : True

Cache đã tồn tại tại: logs/metadata_cache.pkl. Bỏ qua bước quét H5.
Đang load tập vocab...
Đang load tập vocab...
Đã load vocab với vocab_size là 71
Đã load vocab với vocab_size là 71
🛠️ TextEncoder: Sử dụng 4 lớp ConvNeXtV2
🛠️ TextEncoder: Sử dụng 4 lớp ConvNeXtV2
Đang đọc trọng số từ file checkpoint /kaggle/input/models/huutrank4ds/viflow-full-gear/pytorch/default/9/viflow_epoch_104.pt!
Đang đọc trọng số từ file checkpoint /kaggle/input/models/huutrank4ds/viflow-full-gear/pytorch/default/9/viflow_